# 🧠 Materi Kuliah Deep Learning
## Pertemuan 5: Transfer Learning & Pretrained Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hendrick02121977/Deep-Learning/blob/main/notebooks/05_Transfer_Learning.ipynb)

---

**Tujuan Pembelajaran:**
- Memahami konsep Transfer Learning dan kapan menggunakannya
- Mengenal strategi fine-tuning
- Mengimplementasikan Transfer Learning dengan VGG16 dan MobileNetV2
- Memahami penggunaan pretrained models untuk Computer Vision
- Implementasi object detection dengan YOLO

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import (
    VGG16, MobileNetV2, ResNet50, EfficientNetB0
)
import os

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU tersedia: {len(gpus) > 0}")

## 1. Apa itu Transfer Learning?

Transfer Learning adalah teknik menggunakan pengetahuan yang dipelajari model dari satu tugas untuk tugas yang berbeda tetapi terkait.

### Analogi
Jika kamu sudah bisa mengendarai mobil, belajar mengendarai motor akan lebih mudah karena ada pengetahuan yang bisa ditransfer (keseimbangan, aturan lalu lintas, koordinasi).

### Mengapa Transfer Learning?
1. **Data terbatas**: Melatih deep network butuh banyak data. Transfer learning memungkinkan training dengan data lebih sedikit.
2. **Waktu & komputasi**: Model pretrained sudah dilatih berhari-hari/minggu. Kita bisa manfaatkan hasilnya.
3. **Performa lebih baik**: Fitur yang dipelajari dari dataset besar (ImageNet) sangat general dan berguna.

### Strategi Transfer Learning

```
Data sedikit                              Data banyak
    │                                          │
    ▼                                          ▼
Freeze semua base         ←────────►   Fine-tune semua layer
model layers, hanya                    (atau dari layer tertentu)
train classifier baru
```

In [ ]:
# Visualisasi konsep Transfer Learning
def visualize_transfer_learning():
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    fig.suptitle('Strategi Transfer Learning', fontsize=13, fontweight='bold')
    
    strategies = [
        {
            'title': 'Feature Extraction\n(Freeze semua base)',
            'layers': [
                ('Conv Block 1', '#4CAF50', False),
                ('Conv Block 2', '#4CAF50', False),
                ('Conv Block 3', '#4CAF50', False),
                ('Global Pool', '#4CAF50', False),
                ('New Dense', '#F44336', True),
                ('New Output', '#F44336', True),
            ]
        },
        {
            'title': 'Partial Fine-tuning\n(Freeze layer awal)',
            'layers': [
                ('Conv Block 1', '#4CAF50', False),
                ('Conv Block 2', '#4CAF50', False),
                ('Conv Block 3', '#FF9800', True),
                ('Global Pool', '#FF9800', True),
                ('New Dense', '#F44336', True),
                ('New Output', '#F44336', True),
            ]
        },
        {
            'title': 'Full Fine-tuning\n(Train semua layer)',
            'layers': [
                ('Conv Block 1', '#FF9800', True),
                ('Conv Block 2', '#FF9800', True),
                ('Conv Block 3', '#FF9800', True),
                ('Global Pool', '#FF9800', True),
                ('New Dense', '#F44336', True),
                ('New Output', '#F44336', True),
            ]
        }
    ]
    
    for ax, strat in zip(axes, strategies):
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_title(strat['title'], fontsize=10, fontweight='bold')
        ax.axis('off')
        
        n = len(strat['layers'])
        for i, (name, color, trainable) in enumerate(strat['layers']):
            y = 0.9 - i * (0.85 / n)
            rect = plt.Rectangle((0.1, y - 0.055), 0.8, 0.09,
                                   color=color, alpha=0.7 if trainable else 0.4)
            ax.add_patch(rect)
            ax.text(0.5, y - 0.01, name, ha='center', va='center', fontsize=8,
                    fontweight='bold' if trainable else 'normal')
            lock = '🔓 Trainable' if trainable else '🔒 Frozen'
            ax.text(0.92, y - 0.01, lock, ha='left', va='center', fontsize=6.5)
    
    # Legend
    import matplotlib.patches as mpatches
    legend_elements = [
        mpatches.Patch(color='#4CAF50', alpha=0.4, label='Pretrained & Frozen'),
        mpatches.Patch(color='#FF9800', alpha=0.7, label='Pretrained & Trainable'),
        mpatches.Patch(color='#F44336', alpha=0.7, label='Baru & Trainable')
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=9,
               bbox_to_anchor=(0.5, 0.02))
    
    plt.tight_layout(rect=[0, 0.08, 1, 1])
    plt.show()

visualize_transfer_learning()

## 2. Menyiapkan Dataset: Flowers

Kita akan menggunakan dataset bunga (5 kelas: daisy, dandelion, roses, sunflowers, tulips) yang tersedia di TensorFlow.

In [ ]:
# Download dataset bunga
import pathlib

dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

# Hitung gambar per kelas
class_names_flowers = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
print("Distribusi Dataset Bunga:")
total = 0
for cls in class_names_flowers:
    count = len(list((data_dir / cls).glob('*.jpg')))
    print(f"  {cls}: {count} gambar")
    total += count
print(f"  Total: {total} gambar")

# Buat tf.data.Dataset
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print(f"\nTraining batches: {len(train_ds)}")
print(f"Validation batches: {len(val_ds)}")

In [ ]:
# Visualisasi dataset
plt.figure(figsize=(12, 6))
plt.suptitle('Contoh Gambar dari Dataset Bunga', fontsize=13, fontweight='bold')

for images, labels in train_ds.take(1):
    for i in range(min(10, len(images))):
        ax = plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names_flowers[labels[i]], fontsize=9)
        plt.axis('off')

plt.tight_layout()
plt.show()

# Normalisasi dan optimasi pipeline
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(AUTOTUNE)

## 3. Transfer Learning dengan MobileNetV2

In [ ]:
# Strategi 1: Feature Extraction (Freeze base model)
print("=" * 50)
print("Strategi 1: Feature Extraction")
print("=" * 50)

# Load MobileNetV2 tanpa top layer
base_model_mob = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze semua layer base model
base_model_mob.trainable = False

# Data augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Build model
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model_mob(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(5, activation='softmax')(x)

model_fe = keras.Model(inputs, outputs, name='MobileNetV2_FeatureExtraction')

print(f"Total parameters       : {model_fe.count_params():,}")
print(f"Trainable parameters   : {sum(p.numpy().size for p in model_fe.trainable_variables):,}")
print(f"Non-trainable params   : {sum(p.numpy().size for p in model_fe.non_trainable_variables):,}")

model_fe.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nMelatih model (Feature Extraction)...")
history_fe = model_fe.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    verbose=1
)

In [ ]:
# Strategi 2: Fine-tuning
print("=" * 50)
print("Strategi 2: Fine-tuning Layer Atas")
print("=" * 50)

# Unfreeze layer terakhir dari base model
base_model_mob.trainable = True

# Freeze semua layer kecuali 30 layer terakhir
for layer in base_model_mob.layers[:-30]:
    layer.trainable = False

fine_tune_layers = sum(1 for l in base_model_mob.layers if l.trainable)
print(f"Layer yang dilatih dalam base model: {fine_tune_layers}")

# Compile ulang dengan learning rate lebih kecil
model_fe.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # LR kecil untuk fine-tuning!
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nFine-tuning...")
history_ft = model_fe.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    verbose=1
)

In [ ]:
# Visualisasi hasil transfer learning
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transfer Learning: Feature Extraction vs Fine-tuning', fontsize=13, fontweight='bold')

# Gabungkan history
acc = history_fe.history['accuracy'] + history_ft.history['accuracy']
val_acc = history_fe.history['val_accuracy'] + history_ft.history['val_accuracy']
loss = history_fe.history['loss'] + history_ft.history['loss']
val_loss = history_fe.history['val_loss'] + history_ft.history['val_loss']

n_fe = len(history_fe.history['accuracy'])

axes[0].plot(acc, label='Train Acc')
axes[0].plot(val_acc, label='Val Acc')
axes[0].axvline(n_fe - 1, color='black', linestyle='--', alpha=0.5, label='Mulai Fine-tuning')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss, label='Train Loss')
axes[1].plot(val_loss, label='Val Loss')
axes[1].axvline(n_fe - 1, color='black', linestyle='--', alpha=0.5, label='Mulai Fine-tuning')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Accuracy akhir Feature Extraction: {history_fe.history['val_accuracy'][-1]:.4f}")
print(f"Accuracy akhir Fine-tuning       : {history_ft.history['val_accuracy'][-1]:.4f}")

## 4. Perbandingan Model Pretrained

Mari kita bandingkan beberapa arsitektur yang tersedia di Keras.

In [ ]:
# Perbandingan ukuran model pretrained
pretrained_models = [
    {'name': 'MobileNetV2',    'params': 3_538_984, 'imagenet_acc': 71.8, 'input': 224},
    {'name': 'ResNet50',       'params': 25_636_712, 'imagenet_acc': 74.9, 'input': 224},
    {'name': 'VGG16',          'params': 138_357_544, 'imagenet_acc': 71.3, 'input': 224},
    {'name': 'InceptionV3',    'params': 23_851_784, 'imagenet_acc': 77.9, 'input': 299},
    {'name': 'EfficientNetB0', 'params': 5_330_571, 'imagenet_acc': 77.1, 'input': 224},
    {'name': 'EfficientNetB7', 'params': 66_658_687, 'imagenet_acc': 84.3, 'input': 600},
    {'name': 'DenseNet121',    'params': 8_062_504, 'imagenet_acc': 75.0, 'input': 224},
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Perbandingan Pretrained Models', fontsize=13, fontweight='bold')

names = [m['name'] for m in pretrained_models]
params = [m['params'] / 1e6 for m in pretrained_models]  # Dalam juta
accs = [m['imagenet_acc'] for m in pretrained_models]

colors_bar = ['#2196F3', '#4CAF50', '#F44336', '#FF9800', '#9C27B0', '#00BCD4', '#795548']

bars = axes[0].barh(names, params, color=colors_bar, alpha=0.8)
axes[0].set_title('Jumlah Parameter (Juta)')
axes[0].set_xlabel('Parameter (Juta)')
axes[0].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars, params):
    axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}M', va='center', fontsize=8)

bars2 = axes[1].barh(names, accs, color=colors_bar, alpha=0.8)
axes[1].set_title('ImageNet Top-1 Accuracy (%)')
axes[1].set_xlabel('Accuracy (%)')
axes[1].set_xlim(65, 88)
axes[1].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars2, accs):
    axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                 f'{val}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print("\nRekomendasi Penggunaan:")
print("  MobileNetV2 / EfficientNetB0  : Mobile apps, resource-constrained")
print("  ResNet50 / DenseNet121         : Performa baik dengan parameter sedang")
print("  EfficientNetB7                 : Performa terbaik jika resource memadai")

## 5. Prediksi dengan Model Pretrained (ImageNet Classes)

Mari kita gunakan model pretrained langsung untuk prediksi tanpa fine-tuning.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import urllib.request
from io import BytesIO
from PIL import Image

# Load MobileNetV2 pretrained
model_pretrained = MobileNetV2(weights='imagenet')

# Download beberapa gambar untuk prediksi
test_images = [
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg', 'Anjing'),
    ('https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg', 'Kucing'),
]

fig, axes = plt.subplots(1, len(test_images), figsize=(14, 5))
if len(test_images) == 1:
    axes = [axes]
fig.suptitle('Prediksi MobileNetV2 (ImageNet Pretrained)', fontsize=12, fontweight='bold')

for ax, (url, true_label) in zip(axes, test_images):
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            img_data = response.read()
        img = Image.open(BytesIO(img_data)).convert('RGB')
        img_resized = img.resize((224, 224))
        img_array = image.img_to_array(img_resized)
        img_preprocessed = preprocess_input(np.expand_dims(img_array, 0))
        
        preds = model_pretrained.predict(img_preprocessed, verbose=0)
        top3 = decode_predictions(preds, top=3)[0]
        
        ax.imshow(img_resized)
        pred_text = '\n'.join([f'{name}: {prob*100:.1f}%' for _, name, prob in top3])
        ax.set_title(f'True: {true_label}\n\nTop-3 Prediksi:\n{pred_text}', fontsize=8)
        ax.axis('off')
    except Exception as e:
        ax.text(0.5, 0.5, f'Error loading\n{true_label}', ha='center', va='center',
                transform=ax.transAxes)
        ax.set_title(f'{true_label}')
        ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Tips dan Best Practices Transfer Learning

### ✅ Do's:
1. **Gunakan learning rate kecil** saat fine-tuning (1e-4 hingga 1e-6)
2. **Mulai dengan feature extraction** dulu, baru fine-tune jika diperlukan
3. **Gunakan data augmentation** untuk menghindari overfitting
4. **Monitor validation loss** untuk early stopping
5. **Pilih model yang sesuai** dengan constraint (ukuran, kecepatan, akurasi)

### ❌ Don'ts:
1. Jangan gunakan learning rate besar saat fine-tuning (menghancurkan pretrained weights)
2. Jangan fine-tune semua layer jika data sangat sedikit
3. Jangan lupa normalisasi input sesuai format yang diharapkan model

## 7. Latihan Mandiri

1. **Latihan 1**: Ganti MobileNetV2 dengan EfficientNetB0. Bandingkan akurasi dan waktu training.

2. **Latihan 2**: Buat dataset sendiri (10 gambar per kelas, 3 kelas berbeda). Terapkan transfer learning.

3. **Latihan 3**: Implementasikan **Grad-CAM** untuk visualisasi area yang diperhatikan model saat membuat prediksi.

4. **Latihan 4**: Gunakan ResNet50 dan coba berbagai strategi fine-tuning. Dokumentasikan hasilnya.

## Ringkasan

✅ **Transfer Learning**: Menggunakan pengetahuan model yang sudah dilatih untuk tugas baru

✅ **Feature Extraction**: Freeze base model, train classifier baru

✅ **Fine-tuning**: Unfreeze sebagian/semua layer dengan LR sangat kecil

✅ **Model Pretrained**: MobileNetV2, ResNet, VGG, EfficientNet, dll

✅ **Best Practice**: LR kecil, data augmentation, early stopping

---

**Pertemuan Berikutnya:** Generative Adversarial Networks (GAN) 🎨